# Phase 6 — Model comparison: EfficientNet-B0 vs ResNet-50 vs ConvNeXt-Tiny

**Nothing is trained here.** This notebook only reads what `01_train_baseline.ipynb` already saved to
`My Drive/ANA_PosNeg/outputs/<model>/round_<k>/`, so it runs in under a minute and needs **no GPU**.

**What it produces**
1. One table with **validation** and **test** scores side by side for every model (mean ± std over the 5 rounds).
2. A round-by-round view (all models used the *same* 5 folds, so rounds can be compared directly).
3. A chart of the same numbers.

**How to read it**
- **Validation** is the score used to *choose* a model. It is slightly flattering, because the best epoch was picked on it — but equally so for every model, so the comparison is fair.
- **Test** is shown alongside as the honest estimate on unseen patients. It should not be the reason a model is chosen.
- A difference smaller than the **± spread** between rounds is not a real difference.

## 1. Connect to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/ANA_PosNeg")

In [ ]:
!pip install -q timm

## 2. Settings

`SEC_PER_EPOCH` was read off the training logs (T4 GPU) — the notebook did not save timings, so they are typed in here.

In [ ]:
MODELS = {                       # folder name on Drive -> name shown in tables
    "efficientnet_b0": "EfficientNet-B0",
    "resnet50": "ResNet-50",
    "convnext_tiny": "ConvNeXt-Tiny",
}
SEC_PER_EPOCH = {"efficientnet_b0": 20, "resnet50": 28, "convnext_tiny": 36}
THRESHOLD = 0.5

OUTPUTS = DRIVE_ROOT / "outputs"
SAVE_DIR = OUTPUTS / "comparison"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

## 3. Read every round of every model

For each round we take `predictions.csv` — the saved best model's raw outputs (**logits**) for the validation
and test images — and recompute the metrics for both. Same metric code as the training notebook.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def compute_metrics(labels, logits, threshold=THRESHOLD):
    labels = np.asarray(labels).astype(int)
    prob = 1 / (1 + np.exp(-np.asarray(logits)))
    pred = (prob >= threshold).astype(int)
    sens = ((pred == 1) & (labels == 1)).sum() / (labels == 1).sum()   # positives caught
    spec = ((pred == 0) & (labels == 0)).sum() / (labels == 0).sum()   # negatives cleared
    return dict(bal_acc=(sens + spec) / 2, sensitivity=sens, specificity=spec,
                auc=roc_auc_score(labels, prob))


rows = []
for name in MODELS:
    for rdir in sorted((OUTPUTS / name).glob("round_*")):
        if not (rdir / "predictions.csv").exists():
            continue                                    # round not finished
        preds = pd.read_csv(rdir / "predictions.csv")
        hist = pd.read_csv(rdir / "history.csv")
        row = dict(model=name, round=int(rdir.name.split("_")[1]),
                   best_epoch=json.loads((rdir / "summary.json").read_text())["best_epoch"],
                   epochs_run=len(hist))
        for split in ("val", "test"):
            part = preds[preds["split"] == split]
            row.update({f"{split}_{a}": b for a, b in compute_metrics(part["label"], part["logit"]).items()})
        rows.append(row)

per_round = pd.DataFrame(rows)
for name in MODELS:
    n = int((per_round["model"] == name).sum()) if len(per_round) else 0
    print(f"{MODELS[name]:16s} {n} of 5 rounds found" + ("" if n == 5 else "   <-- incomplete!"))
per_round.to_csv(SAVE_DIR / "per_round.csv", index=False)

## 4. Model size

Counted from the actual networks (with our 1-output head), so the numbers are exact rather than quoted.

In [ ]:
import timm

PARAMS_M = {}
for name in MODELS:
    net = timm.create_model(name, pretrained=False, num_classes=1)      # no weights downloaded
    PARAMS_M[name] = sum(p.numel() for p in net.parameters()) / 1e6
    print(f"{MODELS[name]:16s} {PARAMS_M[name]:5.1f} million parameters")

## 5. The comparison table

Sorted by **validation balanced accuracy** (the selection criterion). Test columns sit next to it.
`mean ± std` = average over the 5 rounds ± how much it moved between rounds.

In [ ]:
def pm(s):
    return f"{s.mean():.3f} ± {s.std():.3f}"


table = []
for name, d in per_round.groupby("model"):
    table.append({
        "Model": MODELS[name],
        "Params (M)": round(PARAMS_M[name], 1),
        "Val bal. acc": pm(d["val_bal_acc"]),
        "Val AUC": pm(d["val_auc"]),
        "Test bal. acc": pm(d["test_bal_acc"]),
        "Test sensitivity": pm(d["test_sensitivity"]),
        "Test specificity": pm(d["test_specificity"]),
        "Test AUC": pm(d["test_auc"]),
        "Epochs run (total)": int(d["epochs_run"].sum()),
        "Train time (min)": round(d["epochs_run"].sum() * SEC_PER_EPOCH[name] / 60),
        "_sort": d["val_bal_acc"].mean(),
    })
table = pd.DataFrame(table).sort_values("_sort", ascending=False).drop(columns="_sort").set_index("Model")
table.to_csv(SAVE_DIR / "model_comparison.csv")
display(table)

## 6. Round by round

Every model was trained and tested on the **same** 5 splits, so each row below is a like-for-like contest.
If the "winner" changes from round to round, the models are not really different.

In [ ]:
order = [n for n in MODELS if n in set(per_round["model"])]
for split, title in (("val", "VALIDATION"), ("test", "TEST")):
    piv = per_round.pivot(index="round", columns="model", values=f"{split}_bal_acc")[order]
    wins = piv.idxmax(axis=1).value_counts().reindex(order, fill_value=0)
    piv = piv.rename(columns=MODELS)
    piv.loc["mean"] = piv.mean()
    print(f"{title} balanced accuracy")
    display(piv.round(3))
    print("rounds won:", {MODELS[n]: int(w) for n, w in wins.items()}, "\n")

## 7. Chart

Small dots = the 5 individual rounds. Big dot with bars = mean ± std.
Overlapping bars between models = no measurable difference.

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"
fig, ax = plt.subplots(figsize=(9, 4.2))
for i, name in enumerate(order):
    d = per_round[per_round["model"] == name]
    for off, col, colour, label in ((-0.16, "val_bal_acc", BLUE, "validation"), (0.16, "test_bal_acc", ORANGE, "test")):
        ax.scatter([i + off] * len(d), d[col], color=colour, alpha=0.35, s=28, linewidths=0)
        ax.errorbar(i + off, d[col].mean(), yerr=d[col].std(), fmt="o", color=colour, ms=8,
                    capsize=5, lw=2, label=label if i == 0 else None)
        ax.annotate(f"{d[col].mean():.3f}", (i + off, d[col].mean()), xytext=(10, -3),
                    textcoords="offset points", fontsize=9)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{MODELS[n]}\n{PARAMS_M[n]:.0f} M parameters" for n in order])
ax.set_xlim(-0.6, len(order) - 0.4)
ax.set_ylabel("balanced accuracy")
ax.set_title("Model comparison over 5 cross-validation rounds (threshold 0.5, uncalibrated)", loc="left")
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.0, 0.5))   # outside the plot: never covers a dot
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR / "model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved to", SAVE_DIR)